### Example Fixed Project

The rest of this tutorial uses pre compiled ORBIT configs that are stored as .yaml files in the '~/configs/ folder. There are load and save methods available in ORBIT for working with .yaml files. These example projects each exhibit different functionalities within ORBIT. Using these examples and combinations of them, most project configurations can be modeled. 

In [1]:
import os
import pandas as pd
from ORBIT import ProjectManager, load_config
import csv

weather = pd.read_csv("data/example_weather.csv", parse_dates=["datetime"])\
            .set_index("datetime")

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.

### Load the project configuration

In [2]:
fixed_config = load_config("configs/COE_2022_fixed_bottom_20_MW.yaml")  # Configs can be loaded with absolute or relative paths

print(type(fixed_config))                                         # They are loaded in as dictionaries.

print(f"Num turbines: {fixed_config['plant']['num_turbines']}")   # Once a configuration is loaded, different parameters can  
print(f"Turbine: {fixed_config['turbine']}")                      # be accessed using dict access.
print(f"\nSite: {fixed_config['site']}")

<class 'dict'>
Num turbines: 30
Turbine: 20MW_generic

Site: {'depth': 34, 'distance': 116, 'distance_to_landfall': 50, 'mean_windspeed': 9.17}


### Phases

This fixed project represents a generic Offshore Wind farm with 50 6MW turbines. It includes 5 design modules and 6 installation modules as seen below. This is a common set of modules to run for a fixed bottom project. This config will model the procurement and installation of monopiles, scour protection, array system, export system, offshore substation and the turbines.

In [3]:
print(f"Design phases: {fixed_config['design_phases']}")
print(f"\nInstall phases: {list(fixed_config['install_phases'].keys())}")

Design phases: ['ArraySystemDesign', 'ExportSystemDesign', 'MonopileDesign', 'ScourProtectionDesign', 'OffshoreSubstationDesign']

Install phases: ['ArrayCableInstallation', 'ExportCableInstallation', 'MonopileInstallation', 'OffshoreSubstationInstallation', 'ScourProtectionInstallation', 'TurbineInstallation']


### Run

This project is always being modeled with the example weather project supplied that is representative of US East Coast wind farm locations.

In [4]:
project = ProjectManager(fixed_config, weather=None)
project.run()

ORBIT library intialized at 'C:\upsizing_ORBIT\ORBIT\library'


### Top Level Outputs

ProjectManager offers several high level result categories:
- Installation CapEx
- System CapEx (procurement of BOS subcomponents)
- Turbine CapEx
- Soft CapEx (project management costs)
- Total CapEx
- Total installation time
- etc.

In [5]:
print(f"Installation CapEx:  {project.installation_capex/1e6:.0f} M")
print(f"System CapEx:        {project.system_capex/1e6:.0f} M")
print(f"Turbine CapEx:       {project.turbine_capex/1e6:.0f} M")
print(f"Soft CapEx:          {project.soft_capex/1e6:.0f} M")
print(f"Total CapEx:        {project.total_capex/1e6:.0f} M")

print(f"\nInstallation Time: {project.installation_time:.0f} h")

Installation CapEx:  150 M
System CapEx:        491 M
Turbine CapEx:       1020 M
Soft CapEx:          326 M
Total CapEx:        2139 M

Installation Time: 8007 h


### CapEx Breakdown

In [6]:
# The breakdown of project costs by module is available  at 'capex_breakdown'
project.capex_breakdown

{'Array System': 30078180.188959997,
 'Export System': 100357800.0,
 'Offshore Substation': 83413425.0,
 'Scour Protection': 8680800,
 'Substructure': 268663884.4907459,
 'Array System Installation': 8669801.228685405,
 'Export System Installation': 9440257.203684013,
 'Offshore Substation Installation': 3361105.2115677316,
 'Scour Protection Installation': 12408767.12328767,
 'Substructure Installation': 46621046.5440339,
 'Turbine Installation': 69862876.71232876,
 'Turbine': 1020000000,
 'Soft': 325896000.0,
 'Project': 151250000.0}

### Installation Actions

In [7]:
df = pd.DataFrame(project.actions)    # The project simulation logs are also available for all modules
df

,cost_multiplier,agent,action,duration,cost,level,time,phase,location,phase_name,site_depth,max_waveheight,max_windspeed,transit_speed,hub_height,per_trip
0,0.5,Array Cable Installation Vessel,Mobilize,72.000000,1.800000e+05,ACTION,0.000000,ArrayCableInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.5,Export Cable Installation Vessel,Mobilize,72.000000,1.800000e+05,ACTION,0.000000,ExportCableInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.5,Export Cable Burial Vessel,Mobilize,72.000000,1.800000e+05,ACTION,0.000000,ExportCableInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Onshore Construction,Onshore Construction,0.000000,4.446058e+06,ACTION,0.000000,ExportCableInstallation,Landfall,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.5,Heavy Lift Vessel,Mobilize,72.000000,7.500000e+05,ACTION,0.000000,OffshoreSubstationInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1902,NaN,WTIV,Attach Blade,3.500000,7.291667e+04,ACTION,3781.551784,TurbineInstallation,NaN,TurbineInstallation,34.0,NaN,NaN,NaN,168.0,NaN
1903,NaN,WTIV,Release Blade,1.000000,2.083333e+04,ACTION,3782.551784,TurbineInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1904,NaN,WTIV,Lift Blade,1.680000,3.500000e+04,ACTION,3784.231784,TurbineInstallation,NaN,TurbineInstallation,34.0,NaN,NaN,NaN,168.0,NaN
1905,NaN,WTIV,Attach Blade,3.500000,7.291667e+04,ACTION,3787.731784,TurbineInstallation,NaN,TurbineInstallation,34.0,NaN,NaN,NaN,168.0,NaN


In [8]:
# These logs can be sorted by phase by using DataFrame operations

turbine_install = df.loc[df['phase']=="TurbineInstallation"]
turbine_install

,cost_multiplier,agent,action,duration,cost,level,time,phase,location,phase_name,site_depth,max_waveheight,max_windspeed,transit_speed,hub_height,per_trip
444,1.0,WTIV,Mobilize,168.000000,3.500000e+06,ACTION,972.925118,TurbineInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
447,NaN,WTIV,Fasten Tower Section,4.000000,8.333333e+04,ACTION,976.925118,TurbineInstallation,NaN,TurbineInstallation,34.0,NaN,NaN,NaN,168.0,NaN
449,NaN,WTIV,Fasten Tower Section,4.000000,8.333333e+04,ACTION,980.925118,TurbineInstallation,NaN,TurbineInstallation,34.0,NaN,NaN,NaN,168.0,NaN
452,NaN,WTIV,Fasten Nacelle,4.000000,8.333333e+04,ACTION,984.925118,TurbineInstallation,NaN,TurbineInstallation,34.0,NaN,NaN,NaN,168.0,NaN
455,NaN,WTIV,Fasten Blade,1.500000,3.125000e+04,ACTION,986.425118,TurbineInstallation,NaN,TurbineInstallation,34.0,NaN,NaN,NaN,168.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1902,NaN,WTIV,Attach Blade,3.500000,7.291667e+04,ACTION,3781.551784,TurbineInstallation,NaN,TurbineInstallation,34.0,NaN,NaN,NaN,168.0,NaN
1903,NaN,WTIV,Release Blade,1.000000,2.083333e+04,ACTION,3782.551784,TurbineInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1904,NaN,WTIV,Lift Blade,1.680000,3.500000e+04,ACTION,3784.231784,TurbineInstallation,NaN,TurbineInstallation,34.0,NaN,NaN,NaN,168.0,NaN
1905,NaN,WTIV,Attach Blade,3.500000,7.291667e+04,ACTION,3787.731784,TurbineInstallation,NaN,TurbineInstallation,34.0,NaN,NaN,NaN,168.0,NaN


In [9]:
# Operations can also be grouped to see a total amount of time spend on each operation

turbine_install.groupby(["action"]).sum()['duration']

action
Attach Blade             315.0
Attach Nacelle           180.0
Attach Tower Section     360.0
Fasten Blade             135.0
Fasten Nacelle           120.0
Fasten Tower Section     240.0
Jackdown                  11.8
Jackup                    11.8
Lift Blade               151.2
Lift Nacelle              50.4
Lift Tower Section        75.6
Mobilize                 168.0
Position Onsite           60.0
Reequip                   60.0
Release Blade             90.0
Release Nacelle           90.0
Release Tower Section    180.0
Transit                  684.4
Name: duration, dtype: float64

In [10]:
turbine_install.groupby(["agent"]).sum()['duration']

agent
WTIV    2983.2
Name: duration, dtype: float64

In [11]:
def installation_durations(df):
    import pandas as pd 
    install = df.phase.unique()
    df_final = pd.DataFrame()
    phase = list()
    duration = list()
    cost = list()
    agent = list()
    action = list()
    
    for i in install:
        df_phase = df[df["phase"] == i]
        agents = df_phase.agent.unique()
        
        #print (agents)
        phase.append(i)
        duration.append(df_phase["time"].max())
        cost.append(df_phase["time"].max()/(8760/12) * 2000000) 
        agent.append('Port')
        action.append('Port')
        for j in agents:
            df_agent = df_phase[df_phase["agent"] == j]
            actions = df_agent.action.unique()
            for k in actions:
                df_action = df_agent[df_agent["action"] == k]
                phase.append(i)
                duration.append(df_action["duration"].sum())
                cost.append(df_action["cost"].sum())
                agent.append(j)
                action.append(k)


    df_final = pd.DataFrame(list(zip(phase, agent, action, duration, cost)),
                   columns =['phase', 'agent', 'action', 'duration', 'cost'])

    df_final["day_rate"] = df_final["cost"] / (df_final["duration"]/(24))

    return df_final

In [12]:
df_1 = installation_durations(turbine_install)
display(df_1)
print(df_1["cost"].sum())
df_1.to_csv("data/turbine_output_20_MW.csv")

,phase,agent,action,duration,cost,day_rate
0,TurbineInstallation,Port,Port,3788.125118,1.037842e+07,65753.424658
1,TurbineInstallation,WTIV,Mobilize,168.000000,3.500000e+06,500000.000000
2,TurbineInstallation,WTIV,Fasten Tower Section,240.000000,5.000000e+06,500000.000000
3,TurbineInstallation,WTIV,Fasten Nacelle,120.000000,2.500000e+06,500000.000000
4,TurbineInstallation,WTIV,Fasten Blade,135.000000,2.812500e+06,500000.000000
5,TurbineInstallation,WTIV,Transit,684.400000,1.425833e+07,500000.000000
6,TurbineInstallation,WTIV,Position Onsite,60.000000,1.250000e+06,500000.000000
7,TurbineInstallation,WTIV,Jackup,11.800000,2.458333e+05,500000.000000
8,TurbineInstallation,WTIV,Release Tower Section,180.000000,3.750000e+06,500000.000000
9,TurbineInstallation,WTIV,Lift Tower Section,75.600000,1.575000e+06,500000.000000


72528424.97956909


In [13]:
monopile_install = df.loc[df['phase']=="MonopileInstallation"]
monopile_install.groupby(["action"]).sum()['duration']

action
Bolt TP                     120.000000
Crane Reequip                60.000000
Drive Monopile               45.000000
Fasten Monopile             360.000000
Fasten Transition Piece     240.000000
Jackdown                     11.800000
Jackup                       11.800000
Lower Monopile                0.132000
Lower TP                     30.000000
Mobilize                    168.000000
Position Onsite              60.000000
Release Monopile             90.000000
Release Transition Piece     60.000000
RovSurvey                    30.000000
Transit                     684.400000
Upend Monopile               26.119175
Name: duration, dtype: float64

In [14]:
df_2 = installation_durations(monopile_install)
display(df_2)
print(df_2["cost"].sum())
df_2.to_csv("data/monopile_output_20_MW.csv")

,phase,agent,action,duration,cost,day_rate
0,MonopileInstallation,Port,Port,2619.251175,7.176031e+06,65753.424658
1,MonopileInstallation,WTIV,Mobilize,168.000000,3.500000e+06,500000.000000
2,MonopileInstallation,WTIV,Fasten Monopile,360.000000,7.500000e+06,500000.000000
3,MonopileInstallation,WTIV,Fasten Transition Piece,240.000000,5.000000e+06,500000.000000
4,MonopileInstallation,WTIV,Transit,684.400000,1.425833e+07,500000.000000
5,MonopileInstallation,WTIV,Position Onsite,60.000000,1.250000e+06,500000.000000
6,MonopileInstallation,WTIV,Jackup,11.800000,2.458333e+05,500000.000000
7,MonopileInstallation,WTIV,RovSurvey,30.000000,6.250000e+05,500000.000000
8,MonopileInstallation,WTIV,Release Monopile,90.000000,1.875000e+06,500000.000000
9,MonopileInstallation,WTIV,Upend Monopile,26.119175,5.441495e+05,500000.000000


48785430.10567774
